In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [19]:
dataset=pd.read_csv(r"C:\Users\sspat\OneDrive\Desktop\projects\Car_Booking_Travel_Project\studio\ml_service\car_booking_training_data_1000.csv")
dataset.head(3)

,Car_Type,Duration_Days,Total_Distance_km,Num_Passengers,Package_Level,Start_Region,Season_Index,Car_Age_Years,Total_Cost
0,Sedan,5,1413,7,VIP,Suburban,2,1,2525.16
1,Luxury,4,1381,4,Basic,Suburban,3,7,1812.39
2,Hatchback,6,2342,1,VIP,Suburban,1,4,2856.47


In [20]:
dataset.isnull().sum()

Car_Type             0
Duration_Days        0
Total_Distance_km    0
Num_Passengers       0
Package_Level        0
Start_Region         0
Season_Index         0
Car_Age_Years        0
Total_Cost           0
dtype: int64

In [21]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Car_Type           1000 non-null   object 
 1   Duration_Days      1000 non-null   int64  
 2   Total_Distance_km  1000 non-null   int64  
 3   Num_Passengers     1000 non-null   int64  
 4   Package_Level      1000 non-null   object 
 5   Start_Region       1000 non-null   object 
 6   Season_Index       1000 non-null   int64  
 7   Car_Age_Years      1000 non-null   int64  
 8   Total_Cost         1000 non-null   float64
dtypes: float64(1), int64(5), object(3)
memory usage: 70.4+ KB


In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib



In [23]:
# 1. Define Features (X) and Target (y)
TARGET_COLUMN = 'Total_Cost'
X = dataset.drop(TARGET_COLUMN, axis=1) # Features
y = dataset[TARGET_COLUMN]            # Target

# Define column types (as identified from data.info())
NUMERICAL_FEATURES = ['Duration_Days', 'Total_Distance_km', 'Num_Passengers', 'Season_Index', 'Car_Age_Years']
CATEGORICAL_FEATURES = ['Car_Type', 'Package_Level', 'Start_Region']

In [24]:
# 2. Define Preprocessing Steps (Encoding and Standardization/Normalization)
numerical_transformer = StandardScaler() # Performs Standardization on numerical data
categorical_transformer = OneHotEncoder(handle_unknown='ignore') # Performs One-Hot Encoding on categorical data

# The ColumnTransformer applies the correct transformation to the correct columns
preprocessor = ColumnTransformer(
    transformers=[
        # Standardization for numerical data
        ('num', numerical_transformer, NUMERICAL_FEATURES),
        
        # One-Hot Encoding for categorical data
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ],
    remainder='passthrough' # Keeps any other columns that weren't listed
)

In [25]:
# 3. Split Data into Training and Testing Sets
# We use 80% for training and 20% for testing (test_size=0.2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [26]:
# 4. Create the Full ML Pipeline
# The pipeline chains the preprocessor with the chosen regression model
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42)) 
])

In [27]:
print("Data splitting and Preprocessing Pipeline setup complete.")
print(f"Training samples (X_train): {X_train.shape[0]} rows")
print(f"Testing samples (X_test): {X_test.shape[0]} rows")

Data splitting and Preprocessing Pipeline setup complete.
Training samples (X_train): 800 rows
Testing samples (X_test): 200 rows


In [28]:
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

In [29]:
MODEL_FILENAME = 'cost_prediction_model.joblib'
print("Starting model training on 80% of the data...")
model_pipeline.fit(X_train, y_train)
print("Training complete.")


Starting model training on 80% of the data...
Training complete.


In [30]:
y_pred = model_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
joblib.dump(model_pipeline, MODEL_FILENAME)

['cost_prediction_model.joblib']

In [32]:
model_pipeline.score(X_test, y_test)*100

96.33030656808225

In [36]:
sample_input = pd.DataFrame([{
    'Car_Type': 'SUV',
    'Duration_Days': 5,
    'Total_Distance_km': 750,
    'Num_Passengers': 4,
    'Package_Level': 'Premium',
    'Start_Region': 'City',
    'Season_Index': 2,
    'Car_Age_Years': 3
}])

# Make prediction
predicted_cost = model_pipeline.predict(sample_input)[0]
print(f"Predicted Total Cost for the sample input: ${predicted_cost:.2f}")

Predicted Total Cost for the sample input: $1454.49
